# Image classification with YOLO26 - Complete guide

**Audience.** You've read `yolo-v0.ipynb`. Now you want the *complete* image classification workflow with YOLO26 — every config knob, every metric, every export option, with explanations.

**Approach.** We use the Rock-Paper-Scissors dataset (3 classes, ~2.5K images) so the runs are quick. Everything you learn transfers directly to ImageNet-scale problems.

**You will learn:**
1. How `yolo-cls` differs from `yolo` (detection) under the hood.
2. The exact dataset folder layout YOLO expects for classification.
3. Every important argument to `model.train()` — what it does and a sane default.
4. How to read training output: `loss`, `top1_acc`, `top5_acc`, the `runs/` directory.
5. Validation, inference, top-K predictions, and confusion matrices.
6. Exporting to ONNX / TFLite / TensorRT for deployment.
7. When YOLO-cls is and isn't a better choice than a hand-rolled Keras CNN.


## 1. Setup

Install the Ultralytics package. It pulls in PyTorch, OpenCV, and the rest of the stack.


In [1]:
# Run once per environment
# uv add ultralytics

from ultralytics import YOLO
import ultralytics

ultralytics.checks()    # prints OS, Python, torch, ultralytics, GPU info

Ultralytics 8.4.46 🚀 Python-3.11.14 torch-2.2.2 CPU (Intel Core i7-9750H 2.60GHz)
Setup complete ✅ (12 CPUs, 16.0 GB RAM, 194.8/465.6 GB disk)


## 2. How is YOLO-cls different from YOLO-detect?

Both share the same **backbone** (the convolutional feature extractor). What differs is the **head**:

| component                  | `yolo26n.pt` (detect)                            | `yolo26n-cls.pt` (classify)               |
|----------------------------|---------------------------------------------------|--------------------------------------------|
| Backbone                   | identical (CSP-style with C3k2 blocks)           | identical                                  |
| Neck                       | PANet-style multi-scale feature fusion           | dropped — single feature map is enough     |
| Head                       | predicts boxes + class per anchor                | global pool → linear → softmax             |
| Output                     | `(N_boxes, 4 + num_classes)`                     | `(num_classes,)` softmax                   |
| Loss                       | box loss + cls loss (ProgLoss/STAL)              | cross-entropy                              |
| Default image size         | 640                                              | 224                                        |

**Practical implication.** A YOLO-cls model is essentially a CNN classifier wearing the YOLO name. The advantage isn't a magic architecture — it's the **ecosystem**: same training API, same data loaders, same export pipeline as the rest of YOLO. If your project might *later* need detection or segmentation, starting with YOLO-cls keeps your tooling consistent.


## 3. The dataset folder layout for classification

YOLO classification uses **`torchvision.datasets.ImageFolder`** semantics — the same convention Keras uses. Folder names are class labels, alphabetically sorted.

```
rps_dataset/
├── train/
│   ├── paper/      *.png
│   ├── rock/       *.png
│   └── scissors/   *.png
├── val/                         (optional but recommended)
│   ├── paper/
│   ├── rock/
│   └── scissors/
└── test/                        (optional, for final reporting)
    ├── paper/
    ├── rock/
    └── scissors/
```

**Key points:**
- The **root path** is what you pass to `data=` (e.g. `data='rps_dataset'`). YOLO finds `train/`, `val/`, `test/` automatically.
- If you only have `train/`, YOLO will use it for training and validation will fail. Always provide at least `train/` and `val/`.
- Classes are auto-discovered. **Alphabetical order = label index.** With folders `paper / rock / scissors`, label 0 = paper, 1 = rock, 2 = scissors.
- Image sizes don't need to match — YOLO resizes / crops them at load time.


## 4. Loading a pretrained YOLO26-cls model

We start from a checkpoint pretrained on **ImageNet** (1000 classes). The classification head is replaced automatically when you call `train()` with a different number of classes — you don't lift a finger.

Why start pretrained? **Transfer learning** is one of the most reliable wins in computer vision:
- ImageNet pretraining teaches the early layers to recognise edges, textures, shapes, parts.
- Your downstream task only needs to learn the final mapping from "high-level features" → "my classes".
- For RPS with 2.5K images, transfer learning gets you from ~94% (training from scratch with our Keras CNN) to ~99.5% in a fraction of the epochs.


In [2]:
model = YOLO('yolo26n-cls.pt')   # nano variant, ~6 MB; auto-downloads on first call

# Inspect the architecture
model.info()

YOLO26n-cls summary: 120 layers, 2,812,104 parameters, 0 gradients, 4.3 GFLOPs


(120, 2812104, 0, 4.2763264)

## 5. Training — every important argument explained

Below is the *full* training call we'll use, followed by a table explaining each argument. The defaults Ultralytics ships are well-tuned, but you should know what each one does.


In [3]:
from pathlib import Path
PROJECT_DIR = Path.cwd() / 'runs' / 'test_classify'

In [4]:
results = model.train(
    data       = 'rps_dataset',   # path to dataset root (with train/val subfolders)
    epochs     = 20,           # full passes over the training set
    imgsz      = 224,          # square input size; default 224 for cls
    batch      = 32,           # per-GPU batch size; -1 for auto
    patience   = 5,            # early stopping: stop if val acc doesn't improve for N epochs
    optimizer  = 'auto',       # YOLO26 default is MuSGD; 'auto' picks per task
    lr0        = 0.01,         # initial learning rate
    lrf        = 0.01,         # final LR = lr0 * lrf  (cosine schedule)
    momentum   = 0.937,
    weight_decay = 0.0005,
    warmup_epochs = 3.0,       # gradual LR ramp-up to avoid instability at start
    augment    = True,         # use built-in classification augmentation
    cache      = False,        # 'ram' or 'disk' to speed up subsequent epochs
    workers    = 8,            # data loader workers (set to # CPU cores)
    project    = str(PROJECT_DIR),   # parent folder for outputs
    name       = 'rps_yolo26n',      # subfolder for THIS run
    exist_ok   = False,        # if True, overwrite existing run folder
    seed       = 0,            # reproducibility
    plots      = True,         # save training curves & confusion matrix to disk
)

Ultralytics 8.4.46 🚀 Python-3.11.14 torch-2.2.2 CPU (Intel Core i7-9750H 2.60GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=rps_dataset, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=rps_yolo26n, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=5, perspective=

### 5.1 The argument cheatsheet

| arg              | default     | what it does                                                                                                |
|------------------|-------------|--------------------------------------------------------------------------------------------------------------|
| `data`           | —           | dataset root path (folder layout above) **or** a built-in name like `'imagenette'`                           |
| `epochs`         | 100         | full passes over `train/`. With early stopping, real number is usually lower.                                |
| `imgsz`          | 224         | square crop size. Larger = more accurate, slower, more memory.                                               |
| `batch`          | 16          | per-GPU batch size. `-1` autosizes based on free GPU memory.                                                 |
| `patience`       | 100         | epochs of no validation improvement before stopping. **Set to 5–20 in practice.**                            |
| `optimizer`      | `'auto'`    | `auto`, `SGD`, `Adam`, `AdamW`, `MuSGD` (YOLO26's hybrid SGD+Muon). `auto` picks the best for the task.       |
| `lr0`            | 0.01        | initial LR. Lower for fine-tuning small datasets (try 0.001).                                                |
| `lrf`            | 0.01        | final-LR multiplier. Cosine schedule: lr drops from `lr0` to `lr0 * lrf`.                                    |
| `momentum`       | 0.937       | for SGD-like optimizers; ignored for Adam.                                                                   |
| `weight_decay`   | 5e-4        | L2 regularisation. Default works for most cases.                                                             |
| `warmup_epochs`  | 3.0         | gradual LR ramp at start to avoid divergence.                                                                |
| `augment`        | True        | enables YOLO's built-in cls augmentation (RandomResizedCrop, flip, colour jitter, etc.).                     |
| `cache`          | False       | `'ram'` or `'disk'` to cache decoded images. Big speedup for small datasets.                                 |
| `workers`        | 8           | DataLoader processes. Set to # of physical CPU cores.                                                        |
| `project / name` | —           | controls where logs are written: `runs/classify/<name>/`.                                                    |
| `seed`           | 0           | reproducibility. Set in production training.                                                                 |
| `plots`          | True        | save loss curves, val confusion matrix, sample batch images to disk.                                         |

There are dozens more (label smoothing, mixup, copy-paste, etc.). The full list lives at <https://docs.ultralytics.com/usage/cfg/>.


## 6. Reading the output: the `runs/` directory

After `model.train()` finishes, you'll find:

```
runs/test_classify/rps_yolo26n/
├── args.yaml                 ← exact config you used (great for reproducibility)
├── results.csv               ← per-epoch metrics: loss, top1_acc, top5_acc, lr, time
├── results.png               ← those metrics, plotted
├── confusion_matrix.png      ← from final validation
├── confusion_matrix_normalized.png
├── train_batch0.jpg, ...     ← sample augmented batches (sanity check augmentation!)
├── val_batch0_labels.jpg
├── val_batch0_pred.jpg
└── weights/
    ├── best.pt               ← checkpoint with the highest val top-1 accuracy
    └── last.pt               ← checkpoint at the final epoch
```

**Always inspect `train_batch0.jpg`.** It shows what the model is *actually* training on after augmentation. Catches 90% of dataset/preprocessing bugs.

**`best.pt` is what you ship.** `last.pt` is mostly useful if you want to resume training.


## 7. Validation

Once trained, validation is one line. The model remembers its training data, so no arguments needed.


In [5]:
metrics = model.val()

print(f'top-1 acc : {metrics.top1:.4f}')
print(f'top-5 acc : {metrics.top5:.4f}')
# metrics.confusion_matrix.matrix is a numpy array you can plot

Ultralytics 8.4.46 🚀 Python-3.11.14 torch-2.2.2 CPU (Intel Core i7-9750H 2.60GHz)
YOLO26n-cls summary (fused): 47 layers, 1,529,867 parameters, 0 gradients, 3.2 GFLOPs
WARNING ⚠️ Dataset 'split=val' not found, using 'split=test' instead.
train: /Users/mac/Desktop/Apr 2026/AI/AI-Afternoon/dl-project/notebooks/rps_dataset/train... found 2520 images in 3 classes ✅ 
val: /Users/mac/Desktop/Apr 2026/AI/AI-Afternoon/dl-project/notebooks/rps_dataset/test... found 372 images in 3 classes ✅ 
test: /Users/mac/Desktop/Apr 2026/AI/AI-Afternoon/dl-project/notebooks/rps_dataset/test... found 372 images in 3 classes ✅ 
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 981.5±187.6 MB/s, size: 82.7 KB)
val: Scanning /Users/mac/Desktop/Apr 2026/AI/AI-Afternoon/dl-project/notebooks/rps_dataset/test... 372 images, 0 corrupt: 100% ━━━━━━━━━━━━ 372/372 70.9Mit/s 0.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 24/24 5.8it/s 4.2s0.2s
                   all          1          1
Speed: 0.0

**Top-1 vs Top-5.** Top-1 is whether the predicted class is correct. Top-5 is whether the correct class is among the model's 5 highest-probability classes. With only 3 classes (RPS), top-5 is trivially 100% — top-5 only becomes informative for ImageNet-scale tasks (1000 classes).


## 8. Inference — single image, batch, top-K


In [6]:
# Single image
results = model('paper01-000.png')
r = results[0]

print('top-1 :', r.names[r.probs.top1], f'({r.probs.top1conf.item():.3f})')
print('top-5 :', [(r.names[i], r.probs.data[i].item()) for i in r.probs.top5])


image 1/1 /Users/mac/Desktop/Apr 2026/AI/AI-Afternoon/dl-project/notebooks/paper01-000.png: 224x224 paper 1.00, scissors 0.00, rock 0.00, 35.4ms
Speed: 8.0ms preprocess, 35.4ms inference, 0.1ms postprocess per image at shape (1, 3, 224, 224)
top-1 : paper (1.000)
top-5 : [('paper', 1.0), ('scissors', 2.3634591173049557e-08), ('rock', 6.338866320731995e-09)]


**The `Results` object — what you get back.** Each item in the returned list is a `Results` object. Useful attributes for classification:

| attribute                | what it is                                                       |
|--------------------------|------------------------------------------------------------------|
| `r.path`                 | file path or source                                              |
| `r.names`                | dict mapping class index → class name                            |
| `r.probs.data`           | tensor of length `num_classes`, the softmax probabilities        |
| `r.probs.top1`           | int, index of the most likely class                              |
| `r.probs.top1conf`       | tensor scalar, confidence of `top1`                              |
| `r.probs.top5`           | list of 5 indices, sorted by probability                         |
| `r.orig_img`             | original image as numpy (BGR)                                    |
| `r.plot()`               | returns annotated numpy image you can show/save                  |
| `r.save(filename=...)`   | save annotated image directly                                    |


## 9. Confusion matrix on the test set

If you built a separate `test/` folder, you can either:
1. Re-run `model.val(data='rps_dataset', split='test')` (Ultralytics handles it), or
2. Roll your own with `sklearn`, which gives more flexibility.


## 10. Exporting for deployment

`model.export()` converts your trained PyTorch checkpoint to a deployment format. One line per format.

| format         | best for                                | call                                  |
|----------------|------------------------------------------|---------------------------------------|
| ONNX           | cross-framework, Triton, ML.NET, web    | `model.export(format='onnx')`         |
| TensorRT       | NVIDIA Jetson / data-centre GPUs        | `model.export(format='engine')`       |
| TFLite         | Android, Raspberry Pi, microcontrollers | `model.export(format='tflite')`       |
| CoreML         | iOS / macOS apps                        | `model.export(format='coreml')`       |
| OpenVINO       | Intel CPUs / iGPUs                      | `model.export(format='openvino')`     |
| TorchScript    | C++ / mobile-libtorch                   | `model.export(format='torchscript')`  |

YOLO26's DFL removal makes these exports cleaner than ever — you get a graph with no Python-side post-processing.


In [ ]:
# Example: export to ONNX
# (Comment out the line you don't want to run; ONNX is the most portable starting point.)

# onnx_path = model.export(format='onnx', dynamic=True, opset=17)
# print('exported to', onnx_path)

## 11. Should I use YOLO-cls or a hand-rolled CNN?

Both are reasonable. Here's the honest comparison:

| dimension                | hand-rolled Keras CNN (our `dl-v1`)            | YOLO26-cls                                          |
|--------------------------|------------------------------------------------|------------------------------------------------------|
| Lines of code            | ~50                                            | ~5                                                   |
| Speed of iteration       | full control, easy to tweak architecture       | one-line train, hard to change architecture          |
| Final accuracy on RPS    | ~94-96% (from scratch)                         | ~99.5% (transfer-learning advantage)                 |
| Educational value        | **high** — you see every layer                 | medium — abstracted away                             |
| Production deployment    | you own the export                             | one-line export to 6+ formats                        |
| If you also need detect/seg later | rebuild from scratch                  | already there                                        |
| Best learning order      | **first**, to understand CNNs                  | **second**, to leverage that understanding faster    |

**Our recommendation for AI students:** train the Keras CNN first (educational), then re-train with YOLO-cls (productive). When the YOLO model crushes it on the same data, you understand exactly why — pretrained ImageNet features + battle-tested training recipe.


## 12. What to read / try next

- **`yolo-v1.ipynb`** — runnable training notebook (this notebook is the *theory*; v1 is the *practice*).
- **`test-yolo.ipynb`** — runnable inference notebook with batch + visualisation.
- Try a bigger size: change `'yolo26n-cls.pt'` → `'yolo26m-cls.pt'`, observe accuracy & latency.
- Try detection: same RPS images, but draw bounding boxes around the hand. Use a labelling tool like [Roboflow](https://roboflow.com) or [LabelStudio](https://labelstud.io).
- Read the *Configuration* page: <https://docs.ultralytics.com/usage/cfg/>
